In [28]:
!pip install requests

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached charset_normalizer-3.5.2-cp312-cp312-macosx_10_13_universal2.whl.metadata (46 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached charset_normalizer-3.5.2-cp312-cp312-macosx_10_13_universal2.whl (367 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [requests]


In [33]:
import requests

for year in range(2026, 2027):
    for month in range(1,13):
        if year == 2026 and month > 7:
            break
        print(f"Downloading data for {year}-{month:02d}...")
        url = f"https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year}_{month}.zip"
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        open(f"ontime_{year}_{month}.zip", "wb").write(r.content)

In [4]:
!pip install snowflake

  Using cached pyyaml-6.0.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (2.4 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.7/992.7 kB 25.7 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.4-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 43.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 49.5 MB/s  0:00:00
Using cached annotated_types-0.8.0-py3-none-any.whl (13 kB)
Using cached typing_inspection-0.4.4-py3-none-any.whl (14 kB)
Using cached pyyaml-6.0.3-cp312-cp312-macosx_11_0_arm64.whl (173 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 57.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 11.4 MB/s  0:00:00eta 0:00:01
  Created wheel for snowflake-connector-python: 

In [2]:
import io, gzip, zipfile, snowflake.connector


In [4]:
import credentials
conn = snowflake.connector.connect(
    user=credentials.credentials['user'],
    password=credentials.credentials['password'],
    account=credentials.credentials['account'],
    warehouse=credentials.credentials['warehouse'],
    database=credentials.credentials['database'],
    schema=credentials.credentials['schema']
)

In [ ]:
import os
for file in os.listdir():
    if file.startswith("ontime_") and file.endswith(".zip") and file == 'ontime_2025_10.zip':
        print(f"Uploading {file} to Snowflake...")
        with open(file, "rb") as f:
            zip_data = f.read()
            with zipfile.ZipFile(io.BytesIO(zip_data)) as z:
                for name in z.namelist():
                    with z.open(name) as csv_file:
                        print(name)
                        csv_data = csv_file.read()
                        conn.cursor().execute(f"PUT file://{name} @staging_data", file_stream=io.BytesIO(csv_data))
                        # conn.cursor().execute(f"COPY INTO ontime FROM @%ontime FILE_FORMAT = (TYPE = 'CSV' FIELD_OPTIONALLY_ENCLOSED_BY = '\"' SKIP_HEADER = 1)")

Uploading ontime_2025_10.zip to Snowflake...
On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2025_10.csv


ProgrammingError: 001003 (42000): SQL compilation error:
syntax error line 1 at position 57 unexpected '('.
syntax error line 1 at position 70 unexpected ')'.
syntax error line 1 at position 79 unexpected '.'.
syntax error line 1 at position 84 unexpected '@staging_data'.